In [50]:
import pandas as pd
import matplotlib.pyplot as plt
import duckdb 

In [51]:
df = pd.read_csv("../data/interim/master_cleaned.csv")
df["date"] = pd.to_datetime(df["date"])

df = df.sort_values(["coin_id", "date"])

df["price_index"] = (
    df.groupby("coin_id")["price"]
    .transform(lambda x: x / x.iloc[0] * 100)
)

df.head()

,date,coin_id,price,volume,market_cap,daily_return_pct,price_7d_ma,volume_7d_ma,price_index
0,2025-04-04,bitcoin,83163.987574,3.659446e+10,1.652537e+12,NaN,NaN,NaN,100.000000
1,2025-04-05,bitcoin,83852.007654,3.564729e+10,1.664235e+12,0.827305,NaN,NaN,100.827305
2,2025-04-06,bitcoin,83595.885502,1.491040e+10,1.656020e+12,-0.305445,NaN,NaN,100.519333
3,2025-04-07,bitcoin,78211.483582,3.614038e+10,1.555325e+12,-6.440989,NaN,NaN,94.044894
4,2025-04-08,bitcoin,79179.292268,8.290975e+10,1.581408e+12,1.237425,NaN,NaN,95.208629


In [38]:
df = pd.read_csv("../data/interim/master_cleaned.csv")
df["date"] = pd.to_datetime(df["date"])

 ## EDA 1 
### Volitilaty: Mesauring the how the prise sways daily.
- High Volitilaty = High Risk

In [57]:
# EDA 1
# Standard deviation of daily returns (volatility)

vol = (
    df.groupby("coin_id")["daily_return_pct"]
    .std()
    .reset_index()
    .sort_values("daily_return_pct", ascending=False)
    .reset_index(drop=True)
)

# convert to percentage + round
vol["volatility_pct"] = vol["daily_return_pct"].round(2)

# keep only clean column
vol = vol[["coin_id", "volatility_pct"]]

# clean index
vol.index = vol.index + 1

vol

,coin_id,volatility_pct
1,dogwifcoin,6.73
2,floki,5.82
3,official-trump,5.33
4,bitcoin,2.31


## EDA 2 
### Average return 

- This EDA shows the average daily return on all 3 meme coins and also for Bitcoin. We can see that the meme coins may generate a highrt return from a daily perspectiv, but they also come with fastly more instabillity.
 


In [56]:
# EDA 2
# Average daily return per coin

avg_return = (
    df.groupby("coin_id")["daily_return_pct"]
    .mean()
    .reset_index()
    .sort_values("daily_return_pct", ascending=False)
    .reset_index(drop=True)
)

# convert to percentage + round
avg_return["avg_return_pct"] = (avg_return["daily_return_pct"] * 100).round(2)

# keep only clean column
avg_return = avg_return[["coin_id", "avg_return_pct"]]

# clean index
avg_return.index = avg_return.index + 1

avg_return

,coin_id,avg_return_pct
1,dogwifcoin,2.01
2,floki,-2.47
3,bitcoin,-3.31
4,official-trump,-19.19


# EDA 3
### Maximum drawdown shows the largest drop from a previous peak


In [55]:
# EDA 3
# Maximum drawdown = largest drop from a previous peak

df_drawdown = df.sort_values(["coin_id", "date"]).copy()

# running max per coin
df_drawdown["cum_max"] = df_drawdown.groupby("coin_id")["price"].cummax()

# drawdown calculation
df_drawdown["drawdown"] = (
    (df_drawdown["price"] - df_drawdown["cum_max"]) /
    df_drawdown["cum_max"]
)

# summary table
drawdown_smry = (
    df_drawdown.groupby("coin_id")["drawdown"]
    .min()
    .reset_index()
    .sort_values("drawdown", ascending=True)
    .reset_index(drop=True)
)

# convert to percentage and round
drawdown_smry["drawdown_pct"] = (drawdown_smry["drawdown"] * 100).round(2)

# keep only clean column
drawdown_smry = drawdown_smry[["coin_id", "drawdown_pct"]]

# clean index
drawdown_smry.index = drawdown_smry.index + 1

drawdown_smry

,coin_id,drawdown_pct
1,dogwifcoin,-87.14
2,floki,-82.49
3,official-trump,-81.95
4,bitcoin,-49.63
